In [59]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Читаем файл

In [60]:
spend_path = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\Spend (Done).xlsx"
spend = pd.read_excel(spend_path)

In [61]:
# Приводим названия колонок к удобному формату
spend.columns = (spend.columns.str.strip().str.lower().str.replace(" ", "_", regex=False))
spend.shape

(20779, 8)

In [62]:
spend.head()

,date,source,campaign,impressions,spend,clicks,adgroup,ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.00,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.00,0,NaN,NaN


In [63]:
spend.dtypes

date           datetime64[ns]
source                 object
campaign               object
impressions             int64
spend                 float64
clicks                  int64
adgroup                object
ad                     object
dtype: object

In [64]:
# Общая статистика по заполненности, типам и уникальности
spend_info = pd.DataFrame({
    "column": spend.columns,
    "non_null": spend.notna().sum().values,
    "missing": spend.isna().sum().values,
    "missing_pct": (spend.isna().mean().values * 100).round(2),
    "dtype": spend.dtypes.astype(str).values,
    "unique_values": spend.nunique(dropna=True).values})

spend_info

,column,non_null,missing,missing_pct,dtype,unique_values
0,date,20779,0,0.00,datetime64[ns],355
1,source,20779,0,0.00,object,14
2,campaign,14785,5994,28.85,object,51
3,impressions,20779,0,0.00,int64,4003
4,spend,20779,0,0.00,float64,2859
5,clicks,20779,0,0.00,int64,552
6,adgroup,13951,6828,32.86,object,24
7,ad,13951,6828,32.86,object,176


In [65]:
# Чтобы быстро понять поля с пропусками campaign, adgroup, ad, и не выводить все значения, а сделать короткую таблицу:
ad_fields_summary = pd.DataFrame({
    "column": ["campaign", "adgroup", "ad"],
    "unique_values": [spend["campaign"].nunique(dropna=True), spend["adgroup"].nunique(dropna=True), spend["ad"].nunique(dropna=True)],
    "missing": [spend["campaign"].isna().sum(), spend["adgroup"].isna().sum(), spend["ad"].isna().sum()],
    "missing_pct": [round(spend["campaign"].isna().mean() * 100, 2), round(spend["adgroup"].isna().mean() * 100, 2), round(spend["ad"].isna().mean() * 100, 2)]})

ad_fields_summary

,column,unique_values,missing,missing_pct
0,campaign,51,5994,28.85
1,adgroup,24,6828,32.86
2,ad,176,6828,32.86


## Первичные выводы по столбцам

In [66]:
source_order = spend["source"].drop_duplicates().tolist()

source_summary = (spend.groupby("source", dropna=False).agg(rows_count=("source", "count"), total_impressions=("impressions", "sum"), total_clicks=("clicks", "sum"),
        total_spend=("spend", "sum")).reset_index())

source_summary["rows_pct"] = (source_summary["rows_count"] / len(spend) * 100).round(2)

source_summary["impressions_pct"] = (source_summary["total_impressions"] / source_summary["total_impressions"].sum() * 100).round(2)

source_summary["clicks_pct"] = (source_summary["total_clicks"] / source_summary["total_clicks"].sum() * 100).round(2)

source_summary["spend_pct"] = (source_summary["total_spend"] / source_summary["total_spend"].sum() * 100).round(2)

source_summary["source"] = pd.Categorical(source_summary["source"], categories=source_order, ordered=True)

source_summary = source_summary.sort_values("source").reset_index(drop=True)

source_summary

,source,rows_count,total_impressions,total_clicks,total_spend,rows_pct,impressions_pct,clicks_pct,spend_pct
0,Google Ads,1428,32752334,248487,57798.60,6.87,64.12,49.85,38.66
1,Facebook Ads,9732,2850200,48133,33754.72,46.84,5.58,9.66,22.57
2,CRM,355,0,7995,0.00,1.71,0.00,1.60,0.00
3,Bloggers,787,738460,14250,13439.00,3.79,1.45,2.86,8.99
4,Youtube Ads,1926,8655978,59061,14633.33,9.27,16.95,11.85,9.79
5,SMM,614,23772,11528,7269.52,2.95,0.05,2.31,4.86
6,Tiktok Ads,3066,5007212,28268,11985.67,14.76,9.80,5.67,8.02
7,Organic,518,0,59128,0.00,2.49,0.00,11.86,0.00
8,Telegram posts,1003,705415,16777,6860.36,4.83,1.38,3.37,4.59
9,Webinar,766,301670,3241,2874.04,3.69,0.59,0.65,1.92


In [67]:
# Проверяем пропуски рекламной детализации по источникам
source_missing_summary = (spend.groupby("source", dropna=False).agg(rows_count=("source", "count"), campaign_missing=("campaign", lambda x: x.isna().sum()),
        adgroup_missing=("adgroup", lambda x: x.isna().sum()), ad_missing=("ad", lambda x: x.isna().sum())).reset_index())

source_missing_summary["campaign_missing_pct"] = (source_missing_summary["campaign_missing"] / source_missing_summary["rows_count"] * 100).round(2)

source_missing_summary["adgroup_missing_pct"] = (source_missing_summary["adgroup_missing"] / source_missing_summary["rows_count"] * 100).round(2)

source_missing_summary["ad_missing_pct"] = (source_missing_summary["ad_missing"] / source_missing_summary["rows_count"] * 100).round(2)

source_missing_summary["source"] = pd.Categorical(source_missing_summary["source"], categories=source_order, ordered=True)

source_missing_summary = source_missing_summary.sort_values("source").reset_index(drop=True)

source_missing_summary

,source,rows_count,campaign_missing,adgroup_missing,ad_missing,campaign_missing_pct,adgroup_missing_pct,ad_missing_pct
0,Google Ads,1428,519,1428,1428,36.34,100.00,100.00
1,Facebook Ads,9732,518,518,518,5.32,5.32,5.32
2,CRM,355,355,355,355,100.00,100.00,100.00
3,Bloggers,787,787,787,787,100.00,100.00,100.00
4,Youtube Ads,1926,497,497,497,25.80,25.80,25.80
5,SMM,614,614,543,543,100.00,88.44,88.44
6,Tiktok Ads,3066,436,436,436,14.22,14.22,14.22
7,Organic,518,518,518,518,100.00,100.00,100.00
8,Telegram posts,1003,1003,1003,1003,100.00,100.00,100.00
9,Webinar,766,332,328,328,43.34,42.82,42.82


### Решение по столбцам
В таблице Spend все столбцы потенциально нужны для дальнейшего анализа маркетинговой эффективности.

`date` используется для анализа расходов во времени.  
`source`, `campaign`, `adgroup`, `ad` описывают рекламный источник, кампанию, группу объявлений и конкретное объявление.  
`impressions`, `clicks`, `spend` нужны для расчета маркетинговых метрик: CTR, CPC, CPM, расходов по каналам и дальнейшего анализа юнит-экономики.

Столбцы не удаляем лишних нет.

## Работаем с типами данных

In [68]:
# Преобразуем дату в datetime
spend["date"] = pd.to_datetime(spend["date"], errors="coerce")
spend["date"].dtype

dtype('<M8[ns]')

In [69]:
# Проверяем, все ли даты корректно преобразовались
print("Пустых дат после преобразования:", spend["date"].isna().sum())
print("Минимальная дата:", spend["date"].min())
print("Максимальная дата:", spend["date"].max())

Пустых дат после преобразования: 0
Минимальная дата: 2023-07-03 00:00:00
Максимальная дата: 2024-06-21 00:00:00


In [70]:
# Проверяем числовые поля
numeric_columns = ["impressions", "spend", "clicks"]
spend[numeric_columns].describe()

,impressions,spend,clicks
count,20779.000000,20779.000000,20779.000000
mean,2458.203475,7.195892,23.990616
std,11442.528075,26.760080,85.245714
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,63.000000,0.580000,1.000000
75%,709.000000,5.750000,12.000000
max,431445.000000,774.000000,2415.000000


In [71]:
# Проверяем отрицательные значения
for col in numeric_columns:
    print(f"{col}: отрицательных значений =", (spend[col] < 0).sum())

impressions: отрицательных значений = 0
spend: отрицательных значений = 0
clicks: отрицательных значений = 0


In [72]:
# Проверяем пропуски в числовых полях
spend[numeric_columns].isna().sum()

impressions    0
spend          0
clicks         0
dtype: int64

## Вывод по типам данных

* Столбец `date` приведен к формату datetime.  
* Числовые столбцы `impressions`, `spend`, `clicks` уже имеют корректный числовой формат.  
* Отрицательных значений и пропусков в числовых полях не обнаружено.

## Работаем с дубликатами

In [73]:
# У Spend нет отдельного id, поэтому сначала проверяем полные дубликаты по всем столбцам
full_duplicates_count = spend.duplicated().sum()
full_duplicates_count

np.int64(917)

In [74]:
# Смотрим как выглядят строки, которые участвуют в полных дублях
full_duplicates = spend[spend.duplicated(keep=False)].sort_values(["date", "source", "campaign", "adgroup", "ad"])
full_duplicates.head()

,date,source,campaign,impressions,spend,clicks,adgroup,ad
753,2023-07-23,Bloggers,NaN,0,0.0,0,NaN,NaN
755,2023-07-23,Bloggers,NaN,0,0.0,0,NaN,NaN
768,2023-07-24,Bloggers,NaN,0,0.0,0,NaN,NaN
789,2023-07-24,Bloggers,NaN,0,0.0,0,NaN,NaN
841,2023-07-25,Bloggers,NaN,0,0.0,0,NaN,NaN


In [75]:
# Считаем, сколько данных задвоено в строках, которые будут удалены
duplicates_to_remove = spend[spend.duplicated(keep="first")]
duplicates_to_remove[["impressions", "spend", "clicks"]].sum()

impressions     0.0
spend           0.0
clicks         46.0
dtype: float64

In [76]:
# Удаляем полные дубликаты
n_before_duplicates = len(spend)
spend = spend.drop_duplicates().reset_index(drop=True)
n_after_duplicates = len(spend)

print("Строк до удаления дублей:", n_before_duplicates)
print("Строк после удаления дублей:", n_after_duplicates)
print("Удалено строк:", n_before_duplicates - n_after_duplicates)

Строк до удаления дублей: 20779
Строк после удаления дублей: 19862
Удалено строк: 917


In [77]:
# Проверяем, остались ли полные дубликаты
spend.duplicated().sum()

np.int64(0)

## Проверяем бизнес-дубликаты

In [78]:
# Бизнес-дубль для Spend — это строки с одинаковой рекламной сущностью date + source + campaign + adgroup + ad. Но если числовые значения отличаются, автоматически удалять такие строки нельзя.
business_duplicate_subset = ["date", "source", "campaign", "adgroup", "ad"]
business_duplicates = spend[spend.duplicated(subset=business_duplicate_subset, keep=False)].sort_values(business_duplicate_subset)
business_duplicates.shape[0]

1927

In [79]:
# Сколько строк было бы удалено, если оставить первую строку в каждой группе
spend.duplicated(subset=business_duplicate_subset, keep="first").sum()

np.int64(1085)

In [80]:
# Проверяем, отличаются ли числовые значения внутри бизнес-дублей
business_duplicate_check = (business_duplicates.groupby(business_duplicate_subset, dropna=False).agg(
        rows_count=("source", "count"),
        impressions_unique=("impressions", "nunique"),
        spend_unique=("spend", "nunique"),
        clicks_unique=("clicks", "nunique"),
        total_impressions=("impressions", "sum"),
        total_spend=("spend", "sum"),
        total_clicks=("clicks", "sum")).reset_index())

business_duplicate_check.head()

,date,source,campaign,adgroup,ad,rows_count,impressions_unique,spend_unique,clicks_unique,total_impressions,total_spend,total_clicks
0,2023-07-12,Telegram posts,NaN,NaN,NaN,2,2,2,2,911,4.25,28
1,2023-07-13,Telegram posts,NaN,NaN,NaN,2,2,2,2,507,4.25,25
2,2023-07-14,Telegram posts,NaN,NaN,NaN,2,2,2,2,6061,24.25,40
3,2023-07-15,Telegram posts,NaN,NaN,NaN,2,2,2,2,904,4.25,24
4,2023-07-16,Telegram posts,NaN,NaN,NaN,2,2,2,2,535,4.25,12


In [81]:
# Сколько групп имеют разные числовые значения
business_duplicate_check[(business_duplicate_check["impressions_unique"] > 1) | (business_duplicate_check["spend_unique"] > 1) | (business_duplicate_check["clicks_unique"] > 1)].shape[0]

842

### Вывод по бизнес-дублям
* После удаления полных дублей дополнительно проверяем повторы на уровне рекламной сущности: `date`, `source`, `campaign`, `adgroup`, `ad`.
* Такие строки не удаляем автоматически, потому что при совпадении рекламной сущности числовые значения `impressions`, `spend`, `clicks` могут отличаться. Это может означать несколько частей выгрузки или разные записи статистики по одной рекламной сущности.
* Для дальнейшей аналитики такие данные лучше агрегировать на нужном уровне, например по `date + source + campaign`, а не удалять строки вручную.

## Работаем с пропусками

In [82]:
# Проверяем пропуски после удаления дублей
spend_missing = pd.DataFrame({
    "column": spend.columns,
    "missing": spend.isna().sum().values,
    "missing_pct": (spend.isna().mean().values * 100).round(2),
    "unique_values": spend.nunique(dropna=True).values})

spend_missing.sort_values("missing_pct", ascending=False)

,column,missing,missing_pct,unique_values
6,adgroup,5911,29.76,24
7,ad,5911,29.76,176
2,campaign,5077,25.56,51
0,date,0,0.00,355
3,impressions,0,0.00,4003
1,source,0,0.00,14
5,clicks,0,0.00,552
4,spend,0,0.00,2859


In [83]:
# Заполняем пропуски в рекламной детализации.
# В Spend пропуски есть в campaign, adgroup и ad.
# Надежно дозаполнить их из других таблиц нельзя, так как Spend - исходник по маркетингу, поэтому используем значение Unknown.

columns_to_fill_unknown = ["campaign", "adgroup", "ad"]

for col in columns_to_fill_unknown:
    missing_before = spend[col].isna().sum()
    spend[col] = spend[col].fillna("Unknown")
    missing_after = spend[col].isna().sum()

    print(f"{col}: было пропусков {missing_before}, стало {missing_after}")

campaign: было пропусков 5077, стало 0
adgroup: было пропусков 5911, стало 0
ad: было пропусков 5911, стало 0


### Нормализуем текстовые поля

In [84]:
text_columns = ["source", "campaign", "adgroup", "ad"]

for col in text_columns:
    spend[col] = spend[col].astype("string").str.strip()

# Проверяем количество уникальных значений после нормализации
spend[text_columns].nunique()

source       14
campaign     52
adgroup      25
ad          177
dtype: int64

### Создаем дополнительные маркетинговые метрики

Для дальнейшего анализа рекламной эффективности добавляем метрики `ctr`, `cpc`, `cpm`.

`ctr` показывает долю кликов от показов (клики / показы).  
`cpc` показывает среднюю стоимость клика (расходы / клики).  
`cpm` показывает стоимость тысячи показов (расходы / показы * 1000).

Если показы или клики равны нулю, значение соответствующей метрики заполнено нулем, чтобы избежать ошибок деления на ноль.

In [85]:
# Используем np.where, чтобы избежать деления на ноль.
spend["ctr"] = np.where(spend["impressions"] > 0, spend["clicks"] / spend["impressions"] * 100, 0)
spend["cpc"] = np.where(spend["clicks"] > 0, spend["spend"] / spend["clicks"], 0)
spend["cpm"] = np.where(spend["impressions"] > 0, spend["spend"] / spend["impressions"] * 1000, 0)
spend[["ctr", "cpc", "cpm"]] = spend[["ctr", "cpc", "cpm"]].round(2)
spend[["impressions", "clicks", "spend", "ctr", "cpc", "cpm"]].head()

,impressions,clicks,spend,ctr,cpc,cpm
0,6,0,0.00,0.0,0.00,0.0
1,4,1,0.01,25.0,0.01,2.5
2,0,0,0.00,0.0,0.00,0.0
3,0,0,0.00,0.0,0.00,0.0
4,0,0,0.00,0.0,0.00,0.0


## Сводная таблица по рекламным источникам

In [86]:
source_order = spend["source"].drop_duplicates().tolist()

source_spend_summary = (spend.groupby("source", dropna=False).agg(rows_count=("source", "count"), total_impressions=("impressions", "sum"),
        total_clicks=("clicks", "sum"), total_spend=("spend", "sum")).reset_index())
source_spend_summary["rows_pct"] = (source_spend_summary["rows_count"] / len(spend) * 100).round(2)
source_spend_summary["impressions_pct"] = (source_spend_summary["total_impressions"] / source_spend_summary["total_impressions"].sum() * 100).round(2)
source_spend_summary["clicks_pct"] = (source_spend_summary["total_clicks"] / source_spend_summary["total_clicks"].sum() * 100).round(2)
source_spend_summary["spend_pct"] = (source_spend_summary["total_spend"] / source_spend_summary["total_spend"].sum() * 100).round(2)
source_spend_summary["ctr"] = np.where(source_spend_summary["total_impressions"] > 0, source_spend_summary["total_clicks"] / source_spend_summary["total_impressions"] * 100, 0)
source_spend_summary["cpc"] = np.where(source_spend_summary["total_clicks"] > 0, source_spend_summary["total_spend"] / source_spend_summary["total_clicks"], 0)
source_spend_summary["cpm"] = np.where(source_spend_summary["total_impressions"] > 0, source_spend_summary["total_spend"] / source_spend_summary["total_impressions"] * 1000, 0)
source_spend_summary[["ctr", "cpc", "cpm"]] = (source_spend_summary[["ctr", "cpc", "cpm"]].round(2))
source_spend_summary["source"] = pd.Categorical(source_spend_summary["source"], categories=source_order, ordered=True)
source_spend_summary = (source_spend_summary.sort_values("source").reset_index(drop=True))
source_spend_summary

,source,rows_count,total_impressions,total_clicks,total_spend,rows_pct,impressions_pct,clicks_pct,spend_pct,ctr,cpc,cpm
0,Google Ads,1266,32752334,248487,57798.60,6.37,64.12,49.85,38.66,0.76,0.23,1.76
1,Facebook Ads,9569,2850200,48133,33754.72,48.18,5.58,9.66,22.57,1.69,0.70,11.84
2,CRM,355,0,7995,0.00,1.79,0.00,1.60,0.00,0.00,0.00,0.00
3,Bloggers,632,738460,14250,13439.00,3.18,1.45,2.86,8.99,1.93,0.94,18.20
4,Youtube Ads,1784,8655978,59061,14633.33,8.98,16.95,11.85,9.79,0.68,0.25,1.69
5,SMM,571,23772,11521,7269.52,2.87,0.05,2.31,4.86,48.46,0.63,305.80
6,Tiktok Ads,2985,5007212,28268,11985.67,15.03,9.80,5.67,8.02,0.56,0.42,2.39
7,Organic,514,0,59089,0.00,2.59,0.00,11.85,0.00,0.00,0.00,0.00
8,Telegram posts,836,705415,16777,6860.36,4.21,1.38,3.37,4.59,2.38,0.41,9.73
9,Webinar,766,301670,3241,2874.04,3.86,0.59,0.65,1.92,1.07,0.89,9.53


In [88]:
# Сводная таблица по пропускам рекламной детализации, которая показывает количество значений `Unknown` после заполнения пропусков по полям `campaign`, `adgroup`, `ad`.
ad_fields_summary = pd.DataFrame({
    "column": ["campaign", "adgroup", "ad"],
    "unique_values": [spend["campaign"].nunique(dropna=True), spend["adgroup"].nunique(dropna=True), spend["ad"].nunique(dropna=True)],
    "unknown_count": [(spend["campaign"] == "Unknown").sum(), (spend["adgroup"] == "Unknown").sum(), (spend["ad"] == "Unknown").sum()],
    "unknown_pct": [round((spend["campaign"] == "Unknown").mean() * 100, 2), round((spend["adgroup"] == "Unknown").mean() * 100, 2), round((spend["ad"] == "Unknown").mean() * 100, 2)]})
ad_fields_summary

,column,unique_values,unknown_count,unknown_pct
0,campaign,52,5077,25.56
1,adgroup,25,5911,29.76
2,ad,177,5911,29.76


## Итоговая проверка после очистки
Эта таблица показывает количество строк, показы, клики, расходы, доли от общего объема, а также CTR, CPC и CPM по каждому рекламному источнику

In [89]:
spend_cleaning_summary = pd.DataFrame({
    "metric": ["Строк после очистки", "Полных дублей", "Пропусков всего", "Уникальных source", "Уникальных campaign", "Уникальных adgroup",
        "Уникальных ad", "Минимальная дата", "Максимальная дата", "Итого показы", "Итого клики", "Итого расходы"],
    "value": [len(spend), spend.duplicated().sum(), spend.isna().sum().sum(), spend["source"].nunique(), spend["campaign"].nunique(),
        spend["adgroup"].nunique(), spend["ad"].nunique(), spend["date"].min(), spend["date"].max(), spend["impressions"].sum(), spend["clicks"].sum(),
        round(spend["spend"].sum(), 2)]})

spend_cleaning_summary

,metric,value
0,Строк после очистки,19862
1,Полных дублей,0
2,Пропусков всего,0
3,Уникальных source,14
4,Уникальных campaign,52
5,Уникальных adgroup,25
6,Уникальных ad,177
7,Минимальная дата,2023-07-03 00:00:00
8,Максимальная дата,2024-06-21 00:00:00
9,Итого показы,51079010


### Перед сохранением Spend была проведена ревизия столбцов. В основной очищенный файл оставлены только исходные очищенные поля: дата, источник, кампания, показы, клики, расходы, группа объявлений и объявление.
Метрики `ctr`, `cpc`, `cpm` не включались в основной clean-файл, так как они относятся к аналитическому этапу и могут быть рассчитаны позже в Python или Power BI. При этом они сохранены в отдельной сводной таблице `source_spend_summary`, где используются для первичной оценки эффективности рекламных источников.

In [90]:
final_spend_columns = ["date", "source", "campaign", "impressions", "spend", "clicks", "adgroup", "ad"]
spend_clean = spend[final_spend_columns].copy()
spend_clean.shape

(19862, 8)

# Сохраняем очищенный файл и сводные таблицы

In [91]:
output_dir = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\дополнительные файлы"
os.makedirs(output_dir, exist_ok=True)
spend_clean.to_csv(os.path.join(output_dir, "spend_clean.csv"), index=False, encoding="utf-8-sig")
spend_clean.to_excel(os.path.join(output_dir, "spend_clean.xlsx"), index=False)
source_spend_summary.to_csv(os.path.join(output_dir, "source_spend_summary.csv"), index=False, encoding="utf-8-sig")
ad_fields_summary.to_csv(os.path.join(output_dir, "spend_ad_fields_summary.csv"), index=False, encoding="utf-8-sig")
spend_cleaning_summary.to_csv(os.path.join(output_dir, "spend_cleaning_summary.csv"), index=False, encoding="utf-8-sig")

## Вывод по очистке Spend

В таблице Spend было 20 779 строк и 8 столбцов.  
Таблица содержит данные рекламных программ и ресурсов: дату, источник, кампанию, показы, клики, расходы, группу объявлений и объявление.

* Все столбцы были оставлены, так как каждый из них может использоваться для дальнейшего анализа маркетинговой эффективности и юнит-экономики.  
Столбец `date` был приведен к формату datetime. Числовые столбцы `impressions`, `spend`, `clicks` уже имели корректный числовой формат. Отрицательных значений и пропусков в числовых полях не обнаружено.

* В данных были найдены и удалены полные дубли. Так как у Spend нет отдельного ID строки, полным дублем считалась строка, где совпадают все значения. Удаление дублей важно, чтобы не завышать показы, клики и рекламные расходы.

* Пропуски были обнаружены в рекламных полях `campaign`, `adgroup`, `ad`. Эти значения не заполнялись из Deals, потому что Spend отражает данные рекламных кабинетов, а Deals — CRM-атрибуцию лидов. Между строкой расходов и конкретной сделкой нет надежного ключа для точного восстановления кампании, группы объявлений или объявления. Поэтому пропуски были заполнены значением `Unknown`.

* Для проверки результатов очистки были созданы сводные таблицы `source_spend_summary` и `ad_fields_summary`.  
`source_spend_summary` показывает количество строк, показы, клики, расходы, доли от общего объема, CTR, CPC и CPM по каждому рекламному источнику.  
`ad_fields_summary` показывает количество и долю значений `Unknown` в полях рекламной детализации.